## Video processing

Objective: 
1. Take the video frames as input 
2. Categorize the frames as trigger_event and non_trigger_event
   a. trigger_event: These can be frames for configured person/activity like: 
       Identified cook, maid, or any person
       Kid playing, sleeping, etc
       electronic items status like ON/OFF incase of no activity in a room 
    b. non_trigger_event: These frames can be of configured person/activity to exclude like:
       Recognizable person like Hitesh/Lokeish
       Empty room with no activity
       
3. Then If trigger_event frames found send them to LLM/ LLM vision to track activity/motion 
4. summarize the frames and send summary/alerts via email/telegram/whatsapp -> we can use openclaw to integrate this as well 

In [2]:
!pip3 install opencv-python google-generativeai
!pip3 install -U google-genai


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [4]:

import cv2
print(f"OpenCV Version: {cv2.__version__}")

from google import genai
print("Gemini SDK loaded successfully!")

OpenCV Version: 4.13.0
Gemini SDK loaded successfully!


##### Better to use gemini-2.0-flash instead of gpt-4o as the main difference is that GPT-4o doesn't "watch" the video file natively like Gemini. Instead, we have to extract frames as images and send them in a "vision" request.

In [7]:
!pip3 install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 12.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.6/318.6 kB 26.7 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [8]:
!pip3 install python-dotenv


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [10]:
import os
import cv2
import base64
from dotenv import load_dotenv
from openai import OpenAI

# 1. Load the .env file
load_dotenv() 

# 2. Get the key from environment
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("❌ OPENAI_API_KEY not found in .env file!")

client = OpenAI(api_key=api_key)

def analyze_video_with_gpt(video_path):
    video = cv2.VideoCapture(video_path)
    base64_frames = []
    
    # Get metadata
    fps = video.get(cv2.CAP_PROP_FPS)
    if fps == 0: fps = 30 # Fallback
    
    curr_frame = 0
    while video.isOpened():
        success, frame = video.read()
        if not success:
            break
        
        # Sample 1 frame every 3 seconds to be cost-efficient
        if curr_frame % int(fps * 3) == 0:
            # Resize for speed & cost (GPT doesn't need 4K for cleaning tasks)
            frame = cv2.resize(frame, (640, 480))
            _, buffer = cv2.imencode(".jpg", frame)
            base64_frames.append(base64.b64encode(buffer).decode("utf-8"))
        
        curr_frame += 1
    video.release()

    print(f"✅ Extracted {len(base64_frames)} frames. Analyzing...")

    # 3. Call GPT-4o
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a home security analyst."},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "Who is in these frames and what are they doing? (Focus on cleaning/cooking activity)"},
                    *[{"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{f}", "detail": "low"}} for f in base64_frames]
                ]
            }
        ]
    )
    return response.choices[0].message.content

# Run
print(analyze_video_with_gpt("hall_camera_clip.mp4"))

✅ Extracted 3 frames. Analyzing...
I'm not able to identify the person in these images, but they are engaged in mopping the floor of a living room. The person is using a mop and a yellow bucket to clean the wooden floor around the sofas and coffee table.
